# 00 — Data acquisition

All three datasets fetched programmatically — no manual downloads, no login required.

**Install once:**
```bash
pip install xenaPython requests pandas numpy pyarrow scikit-learn tqdm
```

| # | Dataset | Method | Size |
|---|---------|--------|------|
| 1 | TCGA pan-cancer RNA-seq + survival | xenaPython | ~700 MB |
| 2 | GDSC2 drug sensitivity | Sanger direct URL | ~50 MB |
| 3 | MSK-MET metastasis cohort | cBioPortal REST API | ~50 MB |

In [1]:
import xenaPython as xena
import pandas as pd
import numpy as np
import requests
import io
import time
from pathlib import Path
from tqdm import tqdm

Path('data').mkdir(exist_ok=True)

E:\PycharmProjects\Machine_learning_for_cancer_therapy_resistance_and_metastasis_prediction\.venv\Lib\site-packages\xenaPython\__init__.py:110: FutureWarning: Possible nested set at position 7
  re.sub(r"^[^[]+[[]([^]]*)[]].*$", r"\1", query, flags=re.DOTALL))


## Dataset 1 — TCGA pan-cancer RNA-seq + survival

In [2]:
host = 'https://pancanatlas.xenahubs.net'
expr_dataset = 'EB++AdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.xena'

samples = xena.dataset_samples(host, expr_dataset, 100000)
print(f'TCGA samples: {len(samples)}')

all_genes = xena.dataset_field(host, expr_dataset)
print(f'Genes in dataset: {len(all_genes)}')

TCGA samples: 11060
Genes in dataset: 20532


In [3]:
# Biologically motivated gene panel — EMT, MYC, PI3K, RAS, p53, immune evasion, stemness
HALLMARK_GENES = list(set([
    'VIM','FN1','CDH2','SNAI1','SNAI2','TWIST1','ZEB1','ZEB2',
    'CDH1','EPCAM','KRT8','KRT18','MMP2','MMP9','MMP14',
    'MYC','CCND1','CCND2','CDK4','CDK6','E2F1','MCM2','PCNA',
    'AKT1','AKT2','PIK3CA','PIK3CB','MTOR','PTEN','TSC1','TSC2',
    'RPS6KB1','EIF4EBP1','RAPTOR','RICTOR',
    'KRAS','NRAS','HRAS','RAF1','BRAF','MAP2K1','MAPK1','MAPK3',
    'DUSP6','SPRY2','ETV4','ETV5',
    'TP53','MDM2','CDKN1A','CDKN2A','ATM','ATR','CHEK1','CHEK2',
    'BRCA1','BRCA2','RAD51','XRCC1',
    'CD274','PDCD1LG2','CD80','CD86','CTLA4','IDO1','TGFB1',
    'TGFB2','VEGFA','IL6','JAK1','JAK2','STAT1','STAT3',
    'SOX2','OCT4','NANOG','KLF4','ALDH1A1','CD44','CD24',
    'PROM1','LGR5','NOTCH1','WNT5A','AXIN2',
    'BCL2','BCL2L1','MCL1','BAX','BAD','BID','CASP3','CASP9',
    'CCNB1','CCNE1','CDC20','BUB1','PLK1','AURKA','AURKB',
]))

genes_to_fetch = [g for g in HALLMARK_GENES if g in all_genes]
print(f'Gene panel: {len(HALLMARK_GENES)} total, {len(genes_to_fetch)} found in TCGA')

Gene panel: 100 total, 98 found in TCGA


In [5]:
BATCH = 20  # much smaller — reliable across all network conditions
MAX_RETRIES = 3

expr_chunks = []

for i in tqdm(range(0, len(genes_to_fetch), BATCH), desc='Fetching expression'):
    batch_genes = genes_to_fetch[i:i+BATCH]
    
    for attempt in range(MAX_RETRIES):
        try:
            data = xena.dataset_fetch(host, expr_dataset, samples, batch_genes)
            chunk = pd.DataFrame(
                np.array(data, dtype=np.float32).T,
                index=samples,
                columns=batch_genes
            )
            expr_chunks.append(chunk)
            time.sleep(0.5)
            break  # success, move to next batch
        except Exception as e:
            print(f'Batch {i} attempt {attempt+1} failed: {e}')
            time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
            if attempt == MAX_RETRIES - 1:
                print(f'Skipping batch {i} after {MAX_RETRIES} failures')

tcga_expr = pd.concat(expr_chunks, axis=1).fillna(0)
print(f'Expression matrix: {tcga_expr.shape}')

Fetching expression: 100%|██████████| 5/5 [03:09<00:00, 37.85s/it]

Expression matrix: (11060, 98)


In [6]:
# Fetch survival + clinical data
surv_dataset = 'Survival_SupplementalTable_S1_20171025_xena_sp'
surv_fields = xena.dataset_field(host, surv_dataset)
print('Survival fields:', surv_fields)

surv_data = xena.dataset_fetch(host, surv_dataset, samples, surv_fields)

tcga_clinical = pd.DataFrame(
    np.array(surv_data, dtype=object).T,
    index=samples,
    columns=surv_fields
)

# Rename using exact confirmed column names
tcga_clinical = tcga_clinical.rename(columns={
    'OS':                                   'os_event',
    'OS.time':                              'os_days',
    'cancer type abbreviation':             'cancer_type',
    'age_at_initial_pathologic_diagnosis':  'age',
    'gender':                               'gender',
    'ajcc_pathologic_tumor_stage':          'stage',
})

tcga_clinical['os_event'] = pd.to_numeric(tcga_clinical['os_event'], errors='coerce')
tcga_clinical['os_days']  = pd.to_numeric(tcga_clinical['os_days'],  errors='coerce')

print(tcga_clinical[['os_event','os_days','cancer_type','age','stage']].head(5))
print(f'Cancer types: {tcga_clinical["cancer_type"].nunique()}')

Survival fields: ['DFI', 'DFI.time', 'DSS', 'DSS.time', 'OS', 'OS.time', 'PFI', 'PFI.time', 'Redaction', '_PATIENT', 'age_at_initial_pathologic_diagnosis', 'ajcc_pathologic_tumor_stage', 'birth_days_to', 'cancer type abbreviation', 'cause_of_death', 'clinical_stage', 'death_days_to', 'gender', 'histological_grade', 'histological_type', 'initial_pathologic_dx_year', 'last_contact_days_to', 'margin_status', 'menopause_status', 'new_tumor_event_dx_days_to', 'new_tumor_event_site', 'new_tumor_event_site_other', 'new_tumor_event_type', 'race', 'residual_tumor', 'sampleID', 'treatment_outcome_first_course', 'tumor_status', 'vital_status']
                 os_event  os_days cancer_type age stage
TCGA-OR-A5J1-01       1.0   1355.0           0  58     0
TCGA-OR-A5J2-01       1.0   1677.0           0  44     1
TCGA-OR-A5J3-01       0.0   2091.0           0  23     2
TCGA-OR-A5J5-01       1.0    365.0           0  30     2
TCGA-OR-A5J6-01       0.0   2703.0           0  29     0
Cancer types: 34


In [7]:
# Align samples and save
common = tcga_expr.index.intersection(tcga_clinical.index)
tcga_expr     = tcga_expr.loc[common]
tcga_clinical = tcga_clinical.loc[common]

tcga_expr.to_parquet('data/tcga_expression.parquet')
tcga_clinical.to_csv('data/tcga_clinical.csv')

print(f'Saved {len(common)} TCGA samples.')
print(tcga_clinical['cancer_type'].value_counts().head(10))

Saved 11060 TCGA samples.
cancer_type
2     1215
11     606
16     576
28     572
9      566
30     555
17     552
22     550
14     529
5      492
Name: count, dtype: int64


## Dataset 2 — GDSC2 drug sensitivity

In [11]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [13]:
ic50_raw = pd.read_excel('data/GDSC2_fitted_dose_response_27Oct23.xlsx', engine='openpyxl')
print(f'GDSC2 shape: {ic50_raw.shape}')
print('Columns:', ic50_raw.columns.tolist())

GDSC2 shape: (242036, 16)
Columns: ['DATASET', 'NLME_RESULT_ID', 'NLME_CURVE_ID', 'CELL_LINE_NAME', 'SANGER_MODEL_ID', 'CANCER_TYPE', 'DRUG_ID', 'DRUG_NAME', 'PUTATIVE_TARGET', 'PATHWAY_NAME', 'MIN_CONC', 'MAX_CONC', 'LN_IC50', 'AUC', 'RMSE', 'Z_SCORE']


In [14]:
# Pivot to cell line x drug matrix
ic50 = ic50_raw.pivot_table(
    index='CELL_LINE_NAME',
    columns='DRUG_NAME',
    values='LN_IC50',
    aggfunc='mean'
)

# Keep drugs with ≥60% cell line coverage, cell lines with ≥50% drug coverage
ic50 = ic50.loc[:, ic50.notna().mean() >= 0.6]
ic50 = ic50.loc[ic50.notna().mean(axis=1) >= 0.5]

# Impute with drug-wise median, then z-score per drug
from sklearn.preprocessing import StandardScaler
ic50_filled = ic50.apply(lambda col: col.fillna(col.median()), axis=0)
ic50_z = pd.DataFrame(
    StandardScaler().fit_transform(ic50_filled),
    index=ic50_filled.index,
    columns=ic50_filled.columns
)

cell_info = (
    ic50_raw[['CELL_LINE_NAME','CANCER_TYPE']]
    .drop_duplicates()
    .set_index('CELL_LINE_NAME')
)

print(f'GDSC2: {ic50_z.shape[0]} cell lines x {ic50_z.shape[1]} drugs')
ic50_z.to_parquet('data/gdsc2_ic50.parquet')
cell_info.to_csv('data/gdsc2_cell_info.csv')
print('Saved.')

GDSC2: 965 cell lines x 270 drugs
Saved.


## Dataset 3 — MSK-MET metastasis cohort (cBioPortal API)

In [15]:
BASE = 'https://www.cbioportal.org/api'
STUDY_ID = 'msk_met_2021'

resp = requests.get(f'{BASE}/studies/{STUDY_ID}', timeout=30)
resp.raise_for_status()
study = resp.json()
print(f'Study: {study["name"]}')
print(f'Samples: {study["allSampleCount"]}')

Study: MSK MetTropism (MSK, Cell 2021)
Samples: 25775


In [16]:
resp = requests.get(
    f'{BASE}/studies/{STUDY_ID}/clinical-data',
    params={'clinicalDataType': 'SAMPLE', 'projection': 'DETAILED'},
    timeout=120
)
resp.raise_for_status()
clinical_raw = resp.json()
print(f'Clinical records: {len(clinical_raw)}')

Clinical records: 1065484


In [17]:
clinical_df   = pd.DataFrame(clinical_raw)
clinical_wide = clinical_df.pivot_table(
    index='sampleId',
    columns='clinicalAttributeId',
    values='value',
    aggfunc='first'
)

print('Attributes found:', clinical_wide.columns.tolist())

# Build metastatic label — check which column exists
if 'SAMPLE_TYPE' in clinical_wide.columns:
    clinical_wide['is_metastatic'] = (
        clinical_wide['SAMPLE_TYPE'].str.lower().str.contains('metastas').astype(int)
    )
elif 'METASTATIC_SITE' in clinical_wide.columns:
    clinical_wide['is_metastatic'] = clinical_wide['METASTATIC_SITE'].notna().astype(int)
else:
    raise ValueError(f'Cannot find metastasis column. Available: {clinical_wide.columns.tolist()}')

print(f'Metastatic: {clinical_wide["is_metastatic"].sum()}')
print(f'Primary:    {(clinical_wide["is_metastatic"] == 0).sum()}')

clinical_wide.to_csv('data/mskmet_clinical.csv')
print('MSK-MET saved.')

Attributes found: ['CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'DMETS_DX_ADRENAL_GLAND', 'DMETS_DX_BILIARY_TRACT', 'DMETS_DX_BLADDER_UT', 'DMETS_DX_BONE', 'DMETS_DX_BOWEL', 'DMETS_DX_BREAST', 'DMETS_DX_CNS_BRAIN', 'DMETS_DX_DIST_LN', 'DMETS_DX_FEMALE_GENITAL', 'DMETS_DX_HEAD_NECK', 'DMETS_DX_INTRA_ABDOMINAL', 'DMETS_DX_KIDNEY', 'DMETS_DX_LIVER', 'DMETS_DX_LUNG', 'DMETS_DX_MALE_GENITAL', 'DMETS_DX_MEDIASTINUM', 'DMETS_DX_OVARY', 'DMETS_DX_PLEURA', 'DMETS_DX_PNS', 'DMETS_DX_SKIN', 'DMETS_DX_UNSPECIFIED', 'FGA', 'FRACTION_GENOME_ALTERED', 'GENE_PANEL', 'IS_DIST_MET_MAPPED', 'METASTATIC_SITE', 'MET_COUNT', 'MET_SITE_COUNT', 'MSI_SCORE', 'MSI_TYPE', 'MUTATION_COUNT', 'ONCOTREE_CODE', 'ORGAN_SYSTEM', 'PRIMARY_SITE', 'SAMPLE_COVERAGE', 'SAMPLE_TYPE', 'SUBTYPE', 'SUBTYPE_ABBREVIATION', 'TMB_NONSYNONYMOUS', 'TUMOR_PURITY']
Metastatic: 10143
Primary:    15632
MSK-MET saved.


In [18]:
print('=' * 50)
print('DATA ACQUISITION COMPLETE')
print('=' * 50)
print(f'TCGA expression:  {tcga_expr.shape}  -> data/tcga_expression.parquet')
print(f'TCGA clinical:    {tcga_clinical.shape}  -> data/tcga_clinical.csv')
print(f'GDSC2 IC50:       {ic50_z.shape}  -> data/gdsc2_ic50.parquet')
print(f'MSK-MET clinical: {clinical_wide.shape}  -> data/mskmet_clinical.csv')
print()
print('Next: 01_representation_learning.ipynb')

DATA ACQUISITION COMPLETE
TCGA expression:  (11060, 98)  -> data/tcga_expression.parquet
TCGA clinical:    (11060, 34)  -> data/tcga_clinical.csv
GDSC2 IC50:       (965, 270)  -> data/gdsc2_ic50.parquet
MSK-MET clinical: (25775, 43)  -> data/mskmet_clinical.csv

Next: 01_representation_learning.ipynb
